# Lecture 6 — Spatial Heterogeneity and Calibration
## 第六讲 —— 空间异质性与校准

**Computational Methods for Heterogeneous-Agent Macro**
**异质性主体宏观的计算方法**

Jeffrey Sun

### Environment
### 运行环境

Activate the project, load `HouseholdStages` plus `Printf` / `Plots`.

激活项目，加载 `HouseholdStages` 以及 `Printf` 和 `Plots`。

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages
using Printf
using Plots

## 1 · The spatial setup
## 1 · 空间设定

Three locations `loc1`, `loc2`, `loc3` with productivities $A_1 > A_2 > A_3$. Each location produces a single, costlessly-tradable, perfectly-substitutable numeraire good with Cobb–Douglas technology
$$Y_j = A_j K_j^{\alpha} L_j^{1-\alpha}.$$
Capital flows freely across locations and with the rest of the world, so the country is small in the world bond market: **$r$ is exogenous**. Within the country, MPK equalization across locations pins $K_j / L_j$ given $(r, A_j)$, and wages then satisfy
$$w_j = (1-\alpha)\, A_j\, \big(\tfrac{\alpha A_j}{r+\delta}\big)^{\alpha/(1-\alpha)}.$$

三个地点 `loc1`、`loc2`、`loc3`，生产率 $A_1 > A_2 > A_3$。每地用 Cobb–Douglas 技术生产同一种无成本可贸易、完全可替代的计价品 $Y_j = A_j K_j^{\alpha} L_j^{1-\alpha}$。资本可在地点间及与世界自由流动，本国在世界债券市场上规模小，因此 **$r$ 外生**。国内地点间 MPK 均等化，由 $(r, A_j)$ 决定 $K_j / L_j$，从而工资 $w_j = (1-\alpha) A_j \big(\alpha A_j/(r+\delta)\big)^{\alpha/(1-\alpha)}$。

**Link to L04.** We close the model differently than L04: $r$ is set by the world bond market, so no $K$-tatonnement is needed. The new outer loop runs on the preference shifters $\alpha$, not on $K$ — calibration to population shares replaces market clearing as the equilibrium condition.

**与 L04 的衔接。**本讲与 L04 的闭合方式不同：$r$ 由世界债券市场外生给定，因此不需要 $K$ 的 tâtonnement。新的外层循环作用在偏好平移 $\alpha$ 上而非 $K$ 上 —— 对人口份额的校准取代了市场出清，作为均衡条件。

Households are heterogeneous in **wealth** and **location** only (no idiosyncratic income shocks). Within a period:

家庭只在**财富**和**地点**两个维度上异质（不含个体收入冲击）。一期内分为三个阶段：

1. **Migration** — draw a Gumbel taste shock over destinations and pick a new location, paying the migration cost.  迁移：在目的地之间抽取 Gumbel 偏好冲击，付迁移成本后选择新地点。
2. **Wealth update** — receive the destination's wage and roll wealth $b \mapsto (1+r)b + w_j$.  财富更新：拿到目的地工资，财富按 $(1+r)b + w_j$ 演化。
3. **Consumption / savings** — pick next-period wealth on the grid.  消费/储蓄：在财富网格上选择下期财富。

The chain is
$$\text{Migration} \circ \text{WealthChange} \circ \text{ConsumptionSavings}.$$

链为 $\text{Migration} \circ \text{WealthChange} \circ \text{ConsumptionSavings}$。

### Parameters and the target shares
### 参数与目标份额

Standard macro calibration. The world rate is $r = 0.03$. Productivities are $(1.20, 1.00, 0.85)$ — `loc1` is the high-wage region. Migration cost $C[i, j]$ is symmetric, with adjacent locations cheaper to move between than distant ones.

The target population shares `s_data = (0.30, 0.30, 0.40)` are synthetic: the *data* say 40% of households live in `loc3` (the lowest-productivity region), so we will need a positive preference shifter on `loc3` to rationalize them. The whole point of the calibration in §6 is to recover that shifter.

标准宏观参数。世界利率 $r = 0.03$。生产率 $(1.20, 1.00, 0.85)$ —— `loc1` 是高工资地区。迁移成本 $C[i, j]$ 对称，相邻地点之间的成本低于较远地点。

目标人口份额 `s_data = (0.30, 0.30, 0.40)` 为合成数据：*数据*显示 40% 家庭住在 `loc3`（生产率最低的地区），要解释这一点需要在 `loc3` 上添加正的偏好平移。第 6 节的校准的全部意义就在于恢复这个平移。

In [ ]:
@kwdef struct SpatialParams3
    β :: Float64 = 0.96
    σ :: Float64 = 1.5
    α :: Float64 = 0.36
    δ :: Float64 = 0.08

    r :: Float64 = 0.03                       # world interest rate (exogenous)

    A :: NTuple{3,Float64} = (1.20, 1.00, 0.85)  # productivities

    # Migration cost matrix C[i, j] (origin → destination).
    C_base :: Matrix{Float64} = [0.0 0.5 1.0;
                                 0.5 0.0 0.5;
                                 1.0 0.5 0.0]

    ε_logit :: Float64 = 1.5                  # Gumbel scale

    # Initial guess for the (α₂, α₃) calibration iterate; α₁ ≡ 0.
    α_init :: NTuple{2,Float64} = (0.0, 0.0)

    N_w   :: Int     = 250
    w_min :: Float64 = 0.0
    w_max :: Float64 = 25.0
end
Base.Broadcast.broadcastable(p::SpatialParams3) = Ref(p)

const s_data = [0.30, 0.30, 0.40]
p = SpatialParams3()
@printf "β=%.2f, σ=%.2f, α=%.2f, δ=%.2f, r=%.4f\n" p.β p.σ p.α p.δ p.r
@printf "A = (%.2f, %.2f, %.2f);  ε_logit = %.2f\n" p.A[1] p.A[2] p.A[3] p.ε_logit
println("target shares = ", s_data)

## 2 · Wages from the world rate
## 2 · 由世界利率决定的工资

`spatial_wage(r, A_j, p)` implements $w_j = (1-\alpha) A_j (\alpha A_j / (r+\delta))^{\alpha/(1-\alpha)}$. Call sites that need an env will use the library `make_env(hh; ...)` and pass the wages plus the calibration iterate $(\alpha_2, \alpha_3)$ — that comes after we've built the chain.

`spatial_wage(r, A_j, p)` 实现 $w_j = (1-\alpha) A_j (\alpha A_j / (r+\delta))^{\alpha/(1-\alpha)}$。在建好家庭链之后，调用方会用库函数 `make_env(hh; ...)` 把工资和校准迭代值 $(\alpha_2, \alpha_3)$ 一起打包成 env。

In [ ]:
spatial_wage(r, A_j, p) = (1 - p.α) * A_j *
    (p.α * A_j / (r + p.δ))^(p.α / (1 - p.α))

w1, w2, w3 = spatial_wage(p.r, p.A[1], p), spatial_wage(p.r, p.A[2], p), spatial_wage(p.r, p.A[3], p)
@printf "Wages: (w1, w2, w3) = (%.4f, %.4f, %.4f)\n" w1 w2 w3
@printf "Ratio w1/w3 = %.3f (productivity ratio A1/A3 = %.3f)\n" w1/w3 p.A[1]/p.A[3]

## 3 · The household block
## 3 · 家庭块

Three stages. The migration stage carries an **amenity closure** that reads the preference shifter $\alpha_j$ from `env`:
$$\text{amenity}(j;\, \texttt{env}) = \begin{cases} 0 & j = \texttt{loc1} \\ \texttt{env.}\alpha_2 & j = \texttt{loc2} \\ \texttt{env.}\alpha_3 & j = \texttt{loc3} \end{cases}$$
Normalizing $\alpha_1 \equiv 0$ pins location utility (only differences are identified). The library materializes the length-3 amenity vector once per backward pass, so the hot path is no slower than a static-vector amenity.

三个阶段。迁移阶段携带一个**便利度闭包**，从 `env` 中读取偏好平移 $\alpha_j$：上式中 $\alpha_1 \equiv 0$（标准化，只有差值可识别）。库会在每次后向计算时把这个长度-3 的便利度向量物化一次，所以热路径不比静态向量便利度慢。

The big shift from the column-subtraction trick of earlier drafts: $\alpha$ is now data on the env rather than data on the Spec. The Berry calibration loop in §5 will vary $(\alpha_2, \alpha_3)$ by **rebuilding env** each iteration — no in-place Spec mutation.

与早期版本的列减法技巧相比，这里的关键变化是：$\alpha$ 现在是 env 上的数据，而不是 Spec 上的数据。第 5 节的 Berry 校准循环每次迭代时**重建 env** 来改变 $(\alpha_2, \alpha_3)$ —— 无需就地修改 Spec。

In [ ]:
_u_crra(c, ::Val{1})           = log(c)
_u_crra(c, ::Val{σv}) where σv = (c^(1 - σv)) / (1 - σv)
u_crra(c, valσ::Val) = c < 0 ? -Inf : _u_crra(c, valσ)

function spatial_household(p::SpatialParams3)
    layout = StateLayout(
        StateAxis(:wealth,   continuous_grid(p.w_min, p.w_max;
                                             length = p.N_w, spacing = :log)),
        StateAxis(:location, categorical([:loc1, :loc2, :loc3])),
    )

    # Amenity closure: α₁ ≡ 0 by normalization; α₂, α₃ read from env.
    amenity = (dest; env) -> dest == :loc1 ? 0.0 :
                             dest == :loc2 ? env.α₂ : env.α₃

    migration = MigrationStage(layout;
        location_axis  = :location,
        migration_cost = p.C_base,
        amenity        = amenity,
        ε              = p.ε_logit,
    )

    receipt = WealthChangeStage(layout;
        wealth_post = function (cell; env)
            w_loc = cell.location == :loc1 ? env.w1 :
                    cell.location == :loc2 ? env.w2 :
                                              env.w3
            return (1 + env.r) * cell.wealth + w_loc
        end,
        wealth_axis = :wealth,
    )

    savings = ConsumptionSavingsStage(layout;
        β               = p.β,
        utility         = (cell, c; env) -> u_crra(c, Val(p.σ)),
        wealth_axis     = :wealth,
        monotone_search = :divide_conquer,
    )

    return define_moments!(migration ∘ receipt ∘ savings;
        K_total = at_end(integrand = (cell; env) -> cell.wealth,                        reduce = sum),
        L1      = at_end(integrand = (cell; env) -> cell.location == :loc1 ? 1.0 : 0.0, reduce = sum),
        L2      = at_end(integrand = (cell; env) -> cell.location == :loc2 ? 1.0 : 0.0, reduce = sum),
        L3      = at_end(integrand = (cell; env) -> cell.location == :loc3 ? 1.0 : 0.0, reduce = sum),
    )
end

hh = spatial_household(p)
dims = layout_size(first(hh.spec.stages).input_layout)
@printf "Layout: wealth %d × location %d = %d cells\n" dims[1] dims[2] prod(dims)

### 3.1 What's inside `hh`? / `hh` 里面有什么？

A quick peek at the migration stage's Spec and Buffer. The Spec carries the cost matrix and the amenity *closure*; the Buffer carries the per-cell choice-probability tensor `choice_prob` — the migration stage's "memory" — repopulated by the log-sum-exp on every `backward!`.

简单看一下迁移阶段的 Spec 与 Buffer。Spec 持有成本矩阵和便利度*闭包*；Buffer 持有逐 cell 的选择概率张量 `choice_prob` —— 这是迁移阶段的"记忆"，每次 `backward!` 都会被对数-求和-指数运算重写一遍。

In [ ]:
mig = hh.spec.stages[1]
mig_buf = hh.buffer.stages[1]
@show typeof(mig)
@show typeof(mig.amenity)                       # closure: (dest; env) -> Real
@show size(mig_buf.kernel.choice_prob)          # (N_w, 3, 3): per (wealth, origin), per destination

## 4 · Solving the household at $\alpha = 0$
## 4 · 在 $\alpha = 0$ 时求解家庭块

With *no* preference shifters, choice probabilities depend only on wages and the migration cost. `loc1` has the highest wage; we expect it to attract the largest population share.

无偏好平移时，选择概率只取决于工资和迁移成本。`loc1` 工资最高，预期吸引最大份额人口。

In [ ]:
function solve_household(hh, p, α₂, α₃; V_init = nothing, Λ_init = nothing)
    env = make_env(hh;
        r  = p.r,
        w1 = spatial_wage(p.r, p.A[1], p),
        w2 = spatial_wage(p.r, p.A[2], p),
        w3 = spatial_wage(p.r, p.A[3], p),
        α₂ = α₂, α₃ = α₃,
    )
    res = isnothing(V_init) ?
        solve_steady_state_given_env!(hh, env) :
        solve_steady_state_given_env!(hh, env; V_init, Λ_init)
    shares = [res.moments.L1, res.moments.L2, res.moments.L3] ./
             (res.moments.L1 + res.moments.L2 + res.moments.L3)
    return (; V = res.V, Λ = res.Λ, env, moments = res.moments, shares,
              vfi_iters = res.history.vfi_iters,
              lambda_iters = res.history.lambda_iters)
end

out0 = solve_household(hh, p, 0.0, 0.0)
@printf "mass conservation: ΣΛ = %.10f\n" sum(out0.Λ)
@printf "VFI %d iters, Λ %d iters\n" out0.vfi_iters out0.lambda_iters
@printf "Population shares (uncalibrated): (%.3f, %.3f, %.3f)\n" out0.shares[1] out0.shares[2] out0.shares[3]
@printf "Total wealth K_total = %.4f\n" out0.moments.K_total

The high-productivity region `loc1` is overpopulated relative to the data; `loc3` is underpopulated. Something is keeping people in `loc3` that the model — with $\alpha = 0$ — does not see. That something is what we will calibrate.

高生产率地区 `loc1` 在模型中的人口超过数据；`loc3` 不足。某个让人留在 `loc3` 的因素，在 $\alpha = 0$ 的模型中是看不见的 —— 这正是我们要校准的对象。

In [ ]:
xs = 1:3
plt_init = plot(title = "Uncalibrated vs target shares", ylabel = "share",
                xticks = (xs, ["loc1", "loc2", "loc3"]),
                ylims = (0.0, 0.5), legend = :topright, size = (640, 360))
bar!(plt_init, xs .- 0.18, out0.shares; bar_width = 0.30, label = "model (α=0)")
bar!(plt_init, xs .+ 0.18, s_data;       bar_width = 0.30, label = "target (data)")

## 5 · Calibration by Berry contraction
## 5 · Berry 收缩校准

The wage-only model misses the data on `loc3`. We need preference shifters $(\alpha_2, \alpha_3)$ — recall $\alpha_1 \equiv 0$ — such that the stationary population shares equal `s_data`. Three steps: understand the map $\alpha \to \text{shares}$, write down its (approximate) inverse, then run the loop.

只看工资的模型在 `loc3` 上对不上数据。我们需要找偏好平移 $(\alpha_2, \alpha_3)$（回忆 $\alpha_1 \equiv 0$），使平稳人口份额等于 `s_data`。分三步：先理解 $\alpha \to$ 份额的映射，再写下它的（近似）逆，最后跑循环。

### 5.1 The map $\alpha \to$ shares
### 5.1 映射 $\alpha \to$ 份额

Intuition: a larger amenity $\alpha_j$ makes destination $j$ more attractive, so more households migrate there in steady state. Concretely, bumping $\alpha_3$ from 0 to +0.5 should shift mass *into* `loc3` and *out of* `loc1` / `loc2`. Let's check.

直觉：便利度 $\alpha_j$ 越大，目的地 $j$ 越有吸引力，稳态下更多家庭迁入。把 $\alpha_3$ 从 0 提到 +0.5，应该把人口从 `loc1`/`loc2` 推向 `loc3`。验证一下。

In [ ]:
out_bump = solve_household(hh, p, 0.0, 0.5)
@printf "α₃ = 0.0:  shares = (%.3f, %.3f, %.3f)\n" out0.shares[1] out0.shares[2] out0.shares[3]
@printf "α₃ = 0.5:  shares = (%.3f, %.3f, %.3f)\n" out_bump.shares[1] out_bump.shares[2] out_bump.shares[3]
@printf "Δ on loc3: %+.3f\n" out_bump.shares[3] - out0.shares[3]

### 5.2 Inverting the map — Berry contraction
### 5.2 反演映射 —— Berry 收缩

In static logit, choice probabilities satisfy $\log s_j = (\delta_j - \bar\delta)/\varepsilon + \text{const}$, where $\delta_j$ contains $\alpha_j$ additively. **Berry (1994)** proved this map admits a contraction inversion:
$$\alpha_j^{(k+1)} = \alpha_j^{(k)} + \varepsilon \cdot \bigl(\log s_j^{\text{data}} - \log s_j^{\text{model}}\bigr).$$
If the model is currently *under*-predicting share $j$, the gap $\log s_j^{\text{data}} - \log s_j^{\text{model}}$ is positive and we push $\alpha_j$ up — exactly what 5.1 said would work. The step size is the Gumbel scale $\varepsilon$, which makes the update unit-free in the logit metric.

静态 logit 中，选择概率满足 $\log s_j = (\delta_j - \bar\delta)/\varepsilon + \text{const}$，其中 $\delta_j$ 加性包含 $\alpha_j$。**Berry (1994)** 证明该映射存在收缩反演（上式）。若模型当前*低估*份额 $j$，则差值为正，把 $\alpha_j$ 提高 —— 正是 5.1 中演示的方向。步长为 Gumbel 尺度 $\varepsilon$，这让更新在 logit 度量下无量纲。

**Note on terminology.** What we are doing is sometimes called *indirect inference* in macro talk because we infer $\alpha$ (unobserved) by matching moments through the model. Strictly, Gourieroux–Monfort–Renault (1993) "indirect inference" uses an *auxiliary* model; here we use the model itself as its own auxiliary. The looser usage is common.

**术语说明。**这里所做的有时被称为*间接推断*（indirect inference），因为我们通过模型匹配矩来推断不可观测的 $\alpha$。严格的 Gourieroux–Monfort–Renault (1993) 间接推断使用*辅助模型*，这里以模型自身充当辅助。这种较宽松的用法在宏观文献中常见。

### 5.3 The dynamic caveat
### 5.3 动态情形下的注意事项

Our model is dynamic: $V_j$ depends on $\alpha$ through expected future migration value, not just through the current period's flow utility. The strict Berry contraction theorem applies to static discrete-choice models and does not transfer verbatim. Two things still go right empirically:

1. **Log-shares are monotone in $\alpha$.** Pushing $\alpha_j$ up does increase share $j$ — both directly (this period's flow utility) and indirectly (future periods see the same amenity), and the indirect channel adds to the direct one rather than fighting it.
2. **The $\varepsilon$-metric step is conservative.** The static-logit step is exact at the fixed point; away from it, the dynamic correction is small in $|\log s_{\text{data}} - \log s_{\text{model}}|^2$, so first-order convergence still holds.

In our 3-location calibration the loop converges in well under 15 iterations to gaps below $5 \times 10^{-3}$. (For the strict theorem, see Berry, Levinsohn, Pakes 1995, p. 854.)

我们的模型是动态的：$V_j$ 通过未来迁移期望值依赖于 $\alpha$，不仅仅是当期 flow 效用。严格的 Berry 收缩定理是为静态离散选择模型证明的，无法逐字推广。实证上仍然有两点保留：

1. **log-份额关于 $\alpha$ 单调。**推高 $\alpha_j$ 同时通过直接（当期 flow）和间接（未来期）两条渠道增加份额 $j$，两者方向一致而非互相抵消。
2. **$\varepsilon$ 度量下步长保守。**静态 logit 步长在不动点上精确；偏离不动点时，动态修正项对 $|\log s_{\text{data}} - \log s_{\text{model}}|^2$ 是小量，因此仍保有一阶收敛性。

在我们的三地点校准里，循环远在 15 次迭代以内就把差距压到 $5 \times 10^{-3}$ 以下。严格定理见 Berry, Levinsohn, Pakes (1995, p. 854)。

In [ ]:
function calibrate_shifters!(hh, p, s_data;
                              damping   = 1.0,
                              tol       = 5e-3,
                              maxiter   = 30,
                              verbosity = 1)
    # α = (α₁, α₂, α₃) with α₁ ≡ 0 by normalization.
    α = Float64[0.0, p.α_init[1], p.α_init[2]]
    V, Λ = nothing, nothing
    α_history   = Vector{Float64}[copy(α)]
    gap_history = Vector{Float64}[]
    last_out    = nothing
    iters       = 0
    converged   = false

    while iters < maxiter
        out = solve_household(hh, p, α[2], α[3]; V_init = V, Λ_init = Λ)
        last_out = out
        V, Λ = out.V, out.Λ

        gap = log.(s_data) .- log.(max.(out.shares, 1e-8))
        push!(gap_history, copy(gap))
        iters += 1

        verbosity > 0 && @printf(
            "  iter %2d: shares = (%.3f, %.3f, %.3f); α = (%.3f, %.3f, %.3f); ‖gap‖∞ = %.4f\n",
            iters, out.shares[1], out.shares[2], out.shares[3],
            α[1], α[2], α[3], maximum(abs.(gap)))

        if maximum(abs.(gap)) < tol
            converged = true
            break
        end

        # Berry contraction: α_j ← α_j + ε · (log s_data − log s_model); re-normalize α[1] = 0.
        α .+= damping * p.ε_logit .* gap
        α .-= α[1]
        push!(α_history, copy(α))
    end

    return (; α, iters, converged, α_history, gap_history,
              shares = last_out.shares, V = last_out.V, Λ = last_out.Λ,
              env = last_out.env, moments = last_out.moments)
end

### Run the calibration
### 运行校准

The loop starts at $\alpha = 0$ (where `loc3` is underpopulated), takes a Berry step, re-solves the household, repeats. Convergence is geometric.

迭代从 $\alpha = 0$ 开始（此时 `loc3` 人口不足），作 Berry 步，重新求解家庭块，重复。收敛速率几何级。

In [ ]:
calib = calibrate_shifters!(hh, p, s_data; verbosity = 1)
@printf "\nFinal α = (%.4f, %.4f, %.4f); converged = %s in %d iterations\n" calib.α[1] calib.α[2] calib.α[3] calib.converged calib.iters
@printf "Final shares: (%.3f, %.3f, %.3f) vs target (%.3f, %.3f, %.3f)\n" calib.shares[1] calib.shares[2] calib.shares[3] s_data[1] s_data[2] s_data[3]

### Convergence diagnostics
### 收敛诊断

The log-share gap collapses geometrically; the shifter trajectory overshoots once on iter 2 (when the contraction is far from the fixed point) and damps in.

log 份额差呈几何级衰减；偏好平移轨迹在第 2 步过冲一次（此时离不动点较远），随后阻尼收敛。

In [ ]:
αs   = hcat(calib.α_history...)
gaps = hcat(calib.gap_history...)

plt_α   = plot(1:size(αs, 2), αs',
               label = ["α[1]" "α[2]" "α[3]"],
               marker = :circle, linewidth = 2,
               xlabel = "calibration iteration", ylabel = "α_j",
               title = "Preference shifter trajectory")

plt_gap = plot(1:size(gaps, 2), maximum.(abs, eachcol(gaps)),
               marker = :circle, linewidth = 2, yaxis = :log,
               xlabel = "calibration iteration", ylabel = "max gap (log scale)",
               title = "‖log s_data − log s_model‖∞", legend = false)

plot(plt_α, plt_gap; layout = (1, 2), size = (900, 360))

### Final shares: uncalibrated, calibrated, target
### 最终份额：未校准 / 校准 / 目标

In [ ]:
xs = 1:3
plt_final = plot(title = "Population shares", ylabel = "share",
                 xticks = (xs, ["loc1", "loc2", "loc3"]),
                 ylims = (0.0, 0.5), legend = :topright, size = (640, 360))
bar!(plt_final, xs .- 0.25, out0.shares;  bar_width = 0.22, label = "uncalibrated (α=0)")
bar!(plt_final, xs,         calib.shares; bar_width = 0.22, label = "calibrated")
bar!(plt_final, xs .+ 0.25, s_data;       bar_width = 0.22, label = "target (data)")

## 6 · Reading the calibrated shifters
## 6 · 解读校准的偏好平移

- $\alpha_1 = 0$ by normalization. `loc1`'s 30% share in the data is *fully explained* by its productivity advantage; it needs no amenity boost.
- $\alpha_2 \approx 0$ (about $-0.013$). `loc2`'s 30% share is also broadly consistent with the wage-only model.
- $\alpha_3 \approx 0.75$. `loc3` houses 40% of households despite having the *lowest* wage. The data are telling us about $0.75$ units of utility per period of "amenity" or unobserved preference that the wage model misses. This is the kind of inference indirect identification gives us: a value for a quantity we cannot see directly.

- $\alpha_1 = 0$（标准化）。数据中 `loc1` 的 30% 份额*完全*由其生产率优势解释，无需便利度加成。
- $\alpha_2 \approx 0$（约 $-0.013$）。`loc2` 的 30% 份额也基本与只看工资的模型一致。
- $\alpha_3 \approx 0.75$。`loc3` 的工资*最低*，却仍占 40% 人口。数据告诉我们：每期约 $0.75$ 个单位的效用，对应工资模型看不到的便利度或未观测偏好。这就是间接识别得到的推断 —— 一个我们无法直接观测的量被给了赋值。

## 7 · A closer look: wealth distribution by location
## 7 · 进一步观察：按地点的财富分布

§5 said $\alpha$ moves *mass* across locations. The dual question — given location, how is wealth distributed? — is where the wage differential shows up. With no income heterogeneity, the only buffer-stock motive is the *possibility* of future migration to a different wage, so we expect tight, location-conditional wealth distributions whose mean tracks the local wage.

§5 说 $\alpha$ 让*人口*在地点间移动。对偶问题 —— 给定地点后，财富如何分布？—— 体现的是工资差异的作用。由于没有收入异质性，唯一的预防性储蓄动机来自*未来可能*迁到不同工资的地点，因此我们预期条件财富分布紧而集中，均值跟随本地工资。

Three summary statistics per location:

每个地点三个汇总统计量：

- **Mean wealth.** Sum-weighted average of $w$ on the wealth grid, conditional on location.  均值。给定地点下，$w$ 在财富网格上按质量加权的平均。
- **Mass at the borrowing constraint.** Conditional probability $\Pr(w = 0 \mid \text{loc})$ — the fraction of households stuck at the lower bound.  约束处质量。给定地点下 $\Pr(w = 0 \mid \text{loc})$，即贴着借贷约束下界的家庭比例。
- **Gini coefficient.** Within-location wealth inequality.  Gini 系数。地点内的财富不平等度。

In [ ]:
"""CLAUDE
Compute the Gini coefficient of a discrete distribution `(values, mass)`
where `mass` need not sum to 1 (it's normalized internally). Values are
sorted ascending; the Lorenz-curve integral is evaluated by the trapezoid
rule on the sorted (cumulative-population, cumulative-wealth) pairs.
"""
function _gini(values::AbstractVector, mass::AbstractVector)
    total = sum(mass)
    total ≤ 0 && return NaN
    perm = sortperm(values)
    v    = values[perm]
    p    = mass[perm] ./ total
    cumpop    = vcat(0.0, cumsum(p))
    cumwealth = vcat(0.0, cumsum(v .* p) ./ sum(v .* p))
    area_under_lorenz = sum(diff(cumpop) .* (cumwealth[2:end] .+ cumwealth[1:end-1]) ./ 2)
    return 1 - 2 * area_under_lorenz
end

wgrid = first(hh.spec.stages).input_layout.axes[1].kind.grid
Λ_ss  = calib.Λ
loc_names = ("loc1", "loc2", "loc3")
loc_wages = (spatial_wage(p.r, p.A[1], p),
             spatial_wage(p.r, p.A[2], p),
             spatial_wage(p.r, p.A[3], p))

println("location |  wage  |  share | mean w  | Pr(w=0) |  Gini")
println("---------|--------|--------|---------|---------|--------")
for j in 1:3
    mass_j  = @view Λ_ss[:, j]
    share_j = sum(mass_j)
    mean_w  = sum(wgrid .* mass_j) / share_j
    p_atbc  = mass_j[1] / share_j
    gini_j  = _gini(wgrid, mass_j)
    @printf "%-8s | %.4f | %.3f  | %.4f  |  %.3f  | %.3f\n" loc_names[j] loc_wages[j] share_j mean_w p_atbc gini_j
end

**Reading the table.** `loc1` is the high-wage region: by §5's calibration it houses only 30% of households, but each one is wealthier (highest mean $w$, smallest constrained mass). `loc3` carries 40% of the population at the *lowest* wage — the amenity $\alpha_3$ pulls households in, and because some of them have low savings buffers and the wage doesn't help them build one, the constrained mass is larger and the Gini is higher. The two channels — $\alpha$ moves *mass*, wage moves *wealth-given-location* — are visible separately in the same picture.

**读表。** `loc1` 是高工资地区：根据 §5 的校准只有 30% 的家庭住在那里，但每户都更富有（最高均值 $w$，最低约束处质量）。`loc3` 在*最低*工资下承载 40% 人口 —— 便利度 $\alpha_3$ 把家庭吸引进来，他们中一部分储蓄缓冲较薄、工资又难以帮他们攒起来，所以约束处质量较大、Gini 较高。两条渠道 —— $\alpha$ 推动*人口*、工资推动*给定地点的财富* —— 在同一张图里清晰可分。

### 7.1 Conditional wealth distribution by location
### 7.1 给定地点的条件财富分布

The marginals $\Lambda[:, j]$ scale with the *population* of location $j$, which makes a naive plot misleading — `loc1`'s lower curve gets misread as "the location is poorer" when it is actually just less populated. We normalize each column to integrate to 1 and plot the *conditional* distribution of wealth given location. Wealth is on a log axis because the grid is log-spaced (dense near zero, coarse at the top).

边际分布 $\Lambda[:, j]$ 随地点 $j$ 的*人口规模*缩放，直接画会有误导 —— `loc1` 曲线偏低容易被读成"该地点更穷"，但其实只是人口少。我们把每列归一化到积分为 1，画*给定地点的条件*财富分布。财富轴用 log 刻度，因为网格本身就是 log 间隔（在零附近密、上端稀）。

Each curve integrates to 1 — these are directly comparable in *shape*, not in *level*.

每条曲线积分为 1 —— 这些分布的*形状*可以直接比较，*水平*不行。

In [ ]:
Λ_cond = Λ_ss ./ sum(Λ_ss; dims = 1)   # normalize each location column to sum to 1
plot(wgrid, Λ_cond;
     labels = ["loc1" "loc2" "loc3"],
     xlims  = (wgrid[2], wgrid[end]),   # skip wgrid[1] = 0 for log scale
     xscale = :log10,
     xlabel = "wealth (log scale)", ylabel = "conditional mass",
     title  = "Wealth | location  (each curve integrates to 1)",
     linewidth = 2, size = (720, 360))

## 8 · Foreshadow
## 8 · 预告

Today's bond market was exogenous: we set $r$ from the world. **L07** brings the asset market back inside the model and adds *aggregate* uncertainty — the Krusell–Smith problem. The migration toolkit you just used will reappear in L08–L10 as one instance of a more general pattern (discrete choice as a stage, with calibration as an outer loop).

今天的债券市场外生：我们从世界利率取 $r$。**第 7 讲** 将资产市场重新纳入模型，并加入*总量*不确定性 —— Krusell–Smith 问题。你刚刚使用的迁移工具将在 L08–L10 中作为更普遍模式（"离散选择作为一个 stage、校准作为一外层循环"）的一种实例重新出现。